In [2]:
from autodistill_grounded_sam import GroundedSAM
from autodistill.detection import CaptionOntology
from pathlib import Path
import sys

_pr = Path(".").resolve()
if str(_pr) not in sys.path:
    sys.path.insert(0, str(_pr))
import groundingdino_transformers_compat
groundingdino_transformers_compat.apply()

import os


os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")




model = GroundedSAM(ontology=CaptionOntology({"init": "init"}))


trying to load grounding dino directly


torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)


final text_encoder_type: bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Por cada fila se usa **`row["class"]`** como clase YOLO en el `.txt` (índice según `ALL_CLASSES`). El texto que ve Grounding DINO es `grounding_prompt_for(clase)` (p. ej. varias clases de barco → `"boat"`). No uses una lista fija tipo `ONTOLOGY_CLASSES = ["car"]`: con un solo prompt, `class_id` siempre es `0` y todas las cajas se etiquetaban mal.

In [3]:
import pandas as pd
from tqdm import tqdm
from autodistill_grounded_sam import GroundedSAM
from autodistill.detection import CaptionOntology

DATASET_PATH = "./datasets/sVOC_FLUXFP8"
df = pd.read_csv(f"{DATASET_PATH}/prompts.csv")


ALL_CLASSES = sorted(df["class"].unique())
CLASS_TO_ID = {c: i for i, c in enumerate(ALL_CLASSES)}

classes_dict ={}
for x in ALL_CLASSES:
    classes_dict[x] = x


classes_dict


GROUNDING_PROMPT_BY_CLASS = {
    "pedestrian": "person",
    "Person_sitting": "person",
    "person_sitting": "person",
    'cruise ship': 'boat',
 'fishing boat': 'boat',
 'human powered boat': 'boat',
 'industrial ship': 'boat',
 'jetski': 'boat',
 'military_ship': 'boat',
 'passenger ship': 'boat',
 'pedal boat': 'boat',
 'sailboat': 'boat',
 'small boat': 'boat',
 'speed_boat': 'boat',
 'tugboat': 'boat'
}

def grounding_prompt_for(label: str) -> str:
    return GROUNDING_PROMPT_BY_CLASS.get(label, GROUNDING_PROMPT_BY_CLASS.get(label.lower(), label))


print(f"Clases ({len(ALL_CLASSES)}): {ALL_CLASSES}")
print(f"Filas / imágenes: {len(df)}")

Clases (20): ['aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor']
Filas / imágenes: 16000


In [ ]:
import cv2
import supervision as sv
import json
import numpy as np
import pandas as pd
from datetime import datetime

path = DATASET_PATH
output_path = os.path.join(path, "labels")
labeled_images_dir = os.path.join(path, "imags_labeled")
json_dir = os.path.join(path, "detections_json")
os.makedirs(output_path, exist_ok=True)
os.makedirs(labeled_images_dir, exist_ok=True)
os.makedirs(json_dir, exist_ok=True)

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

_ID_COL = "Unnamed: 0"


def row_image_id(index, row) -> int:
    if _ID_COL in row.index:
        return int(row[_ID_COL])
    return int(index)


for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    try:
        img_id = row_image_id(index, row)
        gt_class = str(row["class"])
        grounding_prompt = grounding_prompt_for(gt_class)

        model.ontology = CaptionOntology({grounding_prompt: gt_class})

        image_path = os.path.join(path, "images", f"{img_id}.png")
        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"No se pudo leer: {image_path}")

        h, w = image.shape[:2]
        txt_path = os.path.join(output_path, f"{img_id}.txt")

        detections = model.predict(image_path)
        detections = detections.with_nms()

        yolo_class_id = CLASS_TO_ID[gt_class]

        record = {
            "index": img_id,
            "dataframe_iter_index": int(index) if isinstance(index, (int, np.integer)) else index,
            "image_path": image_path,
            "image_width": w,
            "image_height": h,
            "ground_truth_class": gt_class,
            "grounding_prompt": grounding_prompt,
            "timestamp": datetime.now().isoformat(),
            "num_detections": len(detections) if detections is not None else 0,
            "detections": [],
        }

        if len(detections) > 0 and detections.xyxy is not None and len(detections.xyxy) > 0:
            confidences = detections.confidence if detections.confidence is not None else np.zeros(len(detections.xyxy))
            vis_labels = [f"{gt_class} {float(conf):.2f}" for conf in confidences]

            for i, (box, conf) in enumerate(zip(detections.xyxy, confidences)):
                x_min, y_min, x_max, y_max = box
                x_center = ((x_min + x_max) / 2) / w
                y_center = ((y_min + y_max) / 2) / h
                bw = (x_max - x_min) / w
                bh = (y_max - y_min) / h

                record["detections"].append({
                    "detection_id": i,
                    "class_name": gt_class,
                    "yolo_class_id": yolo_class_id,
                    "confidence": float(conf),
                    "bbox_xyxy": [float(x_min), float(y_min), float(x_max), float(y_max)],
                    "bbox_yolo": [float(x_center), float(y_center), float(bw), float(bh)],
                    "bbox_width_px": float(x_max - x_min),
                    "bbox_height_px": float(y_max - y_min),
                })

            with open(os.path.join(json_dir, f"{img_id}.json"), "w") as jf:
                json.dump(record, jf, indent=2)

            annotated_image = box_annotator.annotate(scene=image.copy(), detections=detections)
            annotated_image = label_annotator.annotate(
                scene=annotated_image,
                detections=detections,
                labels=vis_labels,
            )

            with open(txt_path, "w") as f:
                for box in detections.xyxy:
                    x_min, y_min, x_max, y_max = box
                    x_center = ((x_min + x_max) / 2) / w
                    y_center = ((y_min + y_max) / 2) / h
                    bw = (x_max - x_min) / w
                    bh = (y_max - y_min) / h
                    f.write(f"{5} {x_center} {y_center} {bw} {bh}\n")
        else:
            with open(os.path.join(json_dir, f"{img_id}.json"), "w") as jf:
                json.dump(record, jf, indent=2)
            annotated_image = image.copy()
            open(txt_path, "w").close()

        cv2.imwrite(os.path.join(labeled_images_dir, f"{img_id}.png"), annotated_image)

    except Exception as err:
        try:
            _iid = row_image_id(index, row)
        except Exception:
            _iid = None
        print(f"Error index={index} img_id={_iid}: {err}")


Processing:   0%|          | 0/16000 [00:00<?, ?it/s]`use_return_dict` is deprecated! Use `return_dict` instead!
torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
None of the inputs have requires_grad=True. Gradients will be None
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
Processing:  17%|█▋        | 2714/16000 [53:27<4:05:14,  1.11s/it] 

Error index=2713 img_id=2713: [Errno 2] No such file or directory: './datasets/sVOC_FLUXFP8/labels/2713.txt'


Processing:  17%|█▋        | 2715/16000 [53:28<4:08:59,  1.12s/it]

Error index=2714 img_id=2714: [Errno 2] No such file or directory: './datasets/sVOC_FLUXFP8/labels/2714.txt'


Processing:  17%|█▋        | 2716/16000 [53:29<4:10:44,  1.13s/it]

...

Processing:  17%|█▋        | 2723/16000 [53:37<4:05:33,  1.11s/it]

Error index=2722 img_id=2722: [Errno 2] No such file or directory: './datasets/sVOC_FLUXFP8/labels/2722.txt'


Processing:  17%|█▋        | 2724/16000 [53:38<4:02:53,  1.10s/it]

Error index=2723 img_id=2723: [Errno 2] No such file or directory: './datasets/sVOC_FLUXFP8/labels/2723.txt'


Processing:  17%|█▋        | 2733/16000 [53:48<4:07:15,  1.12s/it]